# HyPER Light (`hyper_light`) and HyPER (`hyper`)

Short examples for the meta-prompt optimizers in `coolprompt.optimizer.hyper`.

**Requirements**
- Install CoolPrompt (editable from repo root is fine): `pip install -e ".[dev]"` or your usual env.
- A LangChain-compatible chat model. Below we use **OpenRouter** via `langchain_openai.ChatOpenAI` (`OPENROUTER_API_KEY` or `OPENAI_API_KEY`).

**Note:** `hyper` runs many model calls per iteration (paraphrases, scoring, feedback, inner meta-prompt). Start with tiny datasets and `n_iterations=1` if you are testing spend.

## 0. Environment and imports

In [13]:
import os
from langchain_openai import ChatOpenAI

from coolprompt.optimizer.autoprompting_method import build_benchmark_context
from coolprompt.optimizer.hyper.hyper import HyPERMethod
from coolprompt.optimizer.hyper.meta_prompt import HyPERLightMethod
from coolprompt.utils.var_validation import validate_method

In [14]:
def make_chat_model() -> ChatOpenAI:
    api_key = os.environ.get("OPENROUTER_API_KEY") or os.environ.get("OPENAI_API_KEY")
    if not api_key:
        raise RuntimeError(
            "Set OPENROUTER_API_KEY or OPENAI_API_KEY in the environment (or in a .env loaded before this cell)."
        )
    base_url = os.environ.get("OPENROUTER_BASE_URL", "https://openrouter.ai/api/v1")
    model = os.environ.get("INTEGRATION_LLM_MODEL", "openai/gpt-4o-mini")
    return ChatOpenAI(
        model=model,
        temperature=0.3,
        max_completion_tokens=1024,
        max_retries=3,
        base_url=base_url,
        api_key=api_key,
    )


llm = make_chat_model()
llm

ChatOpenAI(client=<openai.resources.chat.completions.completions.Completions object at 0x37f151f90>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x37f153890>, root_client=<openai.OpenAI object at 0x37f1534d0>, root_async_client=<openai.AsyncOpenAI object at 0x37f153610>, model_name='openai/gpt-4o-mini', temperature=0.3, model_kwargs={}, openai_api_key=SecretStr('**********'), openai_api_base='https://openrouter.ai/api/v1', max_retries=3, max_tokens=1024)

## 1. HyPER Light — one meta-prompt step (`hyper_light`)

Single LLM call: the model returns an improved prompt inside `<result_prompt>...</result_prompt>`.

In [15]:
method = HyPERLightMethod()
assert method.name == "hyper_light"

improved = method.optimize(
    model=llm,
    initial_prompt=(
        "You classify movie reviews as POSITIVE or NEGATIVE. "
        "Reply with only the label."
    ),
    problem_description="Binary sentiment on short English reviews.",
    hyper_meta_info={
        "style": "concise",
        "output_contract": "single token: POSITIVE or NEGATIVE",
    },
)

print(improved[:2000])


# Role
You are a sentiment analysis expert specializing in classifying text reviews.

# Task context
The task involves determining the sentiment of short English movie reviews, categorizing them as either POSITIVE or NEGATIVE.

# Instructions
- Analyze the provided movie review.
- Classify the sentiment as either POSITIVE or NEGATIVE based on the content of the review.
- Respond with only the label, without any additional commentary or explanation.

# Output requirements
- The response must be a single token: POSITIVE or NEGATIVE.



### Registered name (same as `PromptTuner` / benchmarks)

In [16]:
impl = validate_method("hyper_light")
impl.name, type(impl).__name__

('hyper_light', 'HyPERLightMethod')

## 2. HyPER — outer loop with mini-batch scoring (`hyper`)

Loads **GSM8K** the same way as YAML benchmarks: `build_benchmark_context` + `dataset.configuration` **`20/20/1`** → **20 train / 20 val** from the train split (40 rows, 50% validation) plus a 1-example test slice. Metric: **exact match (em)**.

The same `llm` runs paraphrases, feedback, inner meta-prompt, and batched answer generation in the evaluator. **BERTScore** (MMR) loads on first use — can take a minute; keep `n_iterations` small when experimenting.

In [20]:
benchmark_config = {
    "dataset": {"name": "gsm8k", "configuration": "30/50/1"},
    "task": "generation",
    "metric": "em",
}

ctx = build_benchmark_context(llm, benchmark_config)
train_x, val_x, train_y, val_y = ctx.dataset_split
print(f"GSM8K split — train={len(train_x)}  val={len(val_x)}  test_slice={len(ctx.test_dataset)}")

hyper = HyPERMethod()
assert hyper.name == "hyper"

starter = (
    "solve the problem "
)

final_prompt = hyper.optimize(
    model=llm,
    initial_prompt=starter,
    dataset_split=ctx.dataset_split,
    evaluator=ctx.evaluator,
    problem_description="math solving",
    hyper_meta_info={
        "problem_description": "math solving"
    },
    n_iterations=1,
    patience=None,
    n_candidates=3,
    top_n_candidates=3,
    k_samples=3,
    mini_batch_size=8,
    contrastive_probability=0.0,
    enable_instance_leak_audit=False,
    random_seed=42,
)

print("--- final prompt ---")
print(final_prompt)

[2026-05-14 16:15:27,037] [INFO] [evaluator.__init__] - Evaluator successfully initialized with em metric
[2026-05-14 16:15:27,038] [INFO] [hyper.optimize] - [HyPER] Starting optimization
[2026-05-14 16:15:27,039] [DEBUG] [hyper.optimize] - [HyPER] Initial prompt: solve the problem ...
[2026-05-14 16:15:27,039] [INFO] [evaluator.evaluate] - Evaluating prompt for generation task on 50 samples


GSM8K split — train=30  val=50  test_slice=1


Evaluating: 100%|██████████| 50/50 [00:26<00:00,  1.88sample/s]
[2026-05-14 16:15:53,575] [INFO] [hyper.optimize] - [HyPER] Initial validation score: 0.9000
[2026-05-14 16:15:53,576] [INFO] [hyper.optimize] - [HyPER] Config: n_iterations=1, patience=None, n_candidates=3, mini_batch_size=8, k_samples=3, top_n_candidates=3
HyPER iterations:   0%|          | 0/1 [00:00<?, ?it/s][2026-05-14 16:15:53,577] [INFO] [hyper.optimize] - 
[2026-05-14 16:15:53,577] [INFO] [hyper.optimize] - [HyPER] === Iteration 1/1 | best_score=0.9000 ===
[2026-05-14 16:15:53,577] [DEBUG] [hyper.optimize] - [HyPER] Current best prompt: solve the problem ...
[2026-05-14 16:15:53,577] [INFO] [hyper.optimize] - [HyPER] Step 1: Generating 3 paraphrase candidates...
[2026-05-14 16:15:55,530] [INFO] [hyper.optimize] - [HyPER] Generated 4 candidates (including original)
[2026-05-14 16:15:55,531] [INFO] [hyper.optimize] - [HyPER] Step 2: Mini-batch sampled: 8 examples (seed=42, indices=[20, 3, 0, 23, 8, 7, 24, 4])
[2026-0

--- final prompt ---

# Role  
You are a skilled problem-solver with a strong analytical mindset, particularly in mathematical problem-solving.

# Task context  
Your task involves addressing a specific math problem that requires critical thinking and strategic planning.

# Instructions  
- Carefully analyze the math problem at hand.
- Develop a clear strategy to tackle it, considering various angles and potential solutions.
- Provide clear step-by-step reasoning and calculations to ensure accuracy and clarity.
- Include a detailed breakdown of calculations at each step to enhance understanding.

# Output requirements  
Present your findings in a structured format, ensuring clarity and coherence in your explanation.

